# Comparative Analysis of Machine Learning Models

Systematic comparison across:
- Multiple model families (linear, tree-based, ensemble, kernel)
- Feature representations (TF-IDF, structured, combined)
- Evaluation dimensions (accuracy, fairness, efficiency)
- Statistical validation (McNemar's test, Friedman test)

## Section 1: Setup

In [ ]:
import pandas as pd
import numpy as np
import re, warnings, time, os, json
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import (train_test_split, GridSearchCV, RandomizedSearchCV,
    StratifiedKFold, cross_val_score)
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, classification_report, confusion_matrix, roc_curve, precision_recall_curve)
from sklearn.preprocessing import LabelEncoder
from xgboost import XGBClassifier
from scipy.sparse import hstack, csr_matrix
import joblib
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
print('Libraries loaded.')

In [ ]:
df = pd.read_csv('../data/cumulative_ai_customer_communication_dataset.csv', low_memory=False)
df['issue_reported_at'] = pd.to_datetime(df['issue_reported_at'], errors='coerce', dayfirst=True)
df['issue_responded'] = pd.to_datetime(df['issue_responded'], errors='coerce', dayfirst=True)
df['target'] = (df['csat_score'] >= 4).astype(int)
print(f'Dataset: {df.shape[0]} rows, Target positive rate: {df["target"].mean()*100:.1f}%')

In [ ]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()
def preprocess_text(text):
    if pd.isna(text) or not isinstance(text, str): return ''
    text = text.lower()
    text = text.encode('ascii','ignore').decode('ascii')
    text = re.sub(r'[^a-z\s]','',text)
    text = re.sub(r'\s+',' ',text).strip()
    tokens = word_tokenize(text)
    tokens = [lemmatizer.lemmatize(t) for t in tokens if t not in stop_words and len(t)>1]
    return ' '.join(tokens)
print('Preprocessing text...')
df['cleaned_message'] = df['customer_message'].apply(preprocess_text)
print(f'Done. Non-empty: {(df["cleaned_message"]!="").sum()}')

In [ ]:
df['response_time_minutes'] = ((df['issue_responded']-df['issue_reported_at']).dt.total_seconds()/60).clip(lower=0).fillna(0)
df['issue_hour'] = df['issue_reported_at'].dt.hour.fillna(0).astype(int)
df['issue_day_of_week'] = df['issue_reported_at'].dt.dayofweek.fillna(0).astype(int)
for col,src in [('channel_encoded','channel_name'),('category_encoded','category'),
                ('subcategory_encoded','sub-category'),('shift_encoded','agent_shift')]:
    le=LabelEncoder(); df[col]=le.fit_transform(df[src].fillna('Unknown'))
tenure_map={'On Job Training':0,'0-30':1,'31-60':2,'61-90':3,'>90':4}
df['tenure_encoded']=df['tenure_bucket'].map(tenure_map).fillna(0).astype(int)
df['has_message']=(df['cleaned_message']!='').astype(int)
df['cleaned_word_count']=df['cleaned_message'].apply(lambda x:len(x.split()) if x else 0)
structured_features=['response_time_minutes','issue_hour','issue_day_of_week','channel_encoded',
    'category_encoded','subcategory_encoded','shift_encoded','tenure_encoded',
    'message_length','word_count','has_message','cleaned_word_count']
print(f'Structured features: {len(structured_features)}')

In [ ]:
X_structured = df[structured_features].fillna(0)
y = df['target']
X_train, X_test, y_train, y_test = train_test_split(X_structured, y, test_size=0.2, random_state=42, stratify=y)
tfidf = TfidfVectorizer(max_features=5000, ngram_range=(1,2), min_df=5)
X_text_train = tfidf.fit_transform(df.loc[X_train.index,'cleaned_message'])
X_text_test = tfidf.transform(df.loc[X_test.index,'cleaned_message'])
X_combined_train = hstack([X_text_train, csr_matrix(X_train.values)])
X_combined_test = hstack([X_text_test, csr_matrix(X_test.values)])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
neg_count=(y_train==0).sum(); pos_count=(y_train==1).sum(); scale_weight=neg_count/pos_count
print(f'Train:{X_train.shape[0]}, Test:{X_test.shape[0]}, TF-IDF:{X_text_train.shape[1]}, Combined:{X_combined_train.shape[1]}')

## Section 2: Train All Model Variants

In [ ]:
# Comprehensive model comparison
print('Training all model variants...')
from sklearn.naive_bayes import MultinomialNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier

results_all = []

# --- Text-based models ---
text_models = {
    'NB (TF-IDF)': MultinomialNB(alpha=1.0),
    'LR (TF-IDF)': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'RF (TF-IDF)': RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42, class_weight='balanced', n_jobs=-1),
    'SVM (TF-IDF)': CalibratedClassifierCV(LinearSVC(max_iter=2000, random_state=42, class_weight='balanced'), cv=3),
}
for name, model in text_models.items():
    start=time.time(); model.fit(X_text_train, y_train); train_time=time.time()-start
    y_pred=model.predict(X_text_test); y_prob=model.predict_proba(X_text_test)[:,1]
    results_all.append({'Model':name,'Feature':'TF-IDF','Accuracy':accuracy_score(y_test,y_pred),
        'Precision':precision_score(y_test,y_pred),'Recall':recall_score(y_test,y_pred),
        'F1':f1_score(y_test,y_pred),'AUC':roc_auc_score(y_test,y_prob),'Time(s)':round(train_time,2)})
    print(f'  {name}: F1={results_all[-1]["F1"]*100:.2f}%')

# --- Structured-feature models ---
struct_models = {
    'LR (Struct)': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'DT (Struct)': DecisionTreeClassifier(max_depth=10, random_state=42, class_weight='balanced'),
    'RF (Struct)': RandomForestClassifier(n_estimators=200, max_depth=20, random_state=42, class_weight='balanced', n_jobs=-1),
    'XGBoost (Struct)': XGBClassifier(n_estimators=300, max_depth=6, learning_rate=0.1, scale_pos_weight=scale_weight, random_state=42, eval_metric='logloss', use_label_encoder=False),
    'GB (Struct)': GradientBoostingClassifier(n_estimators=200, max_depth=5, learning_rate=0.1, random_state=42),
}
for name, model in struct_models.items():
    start=time.time(); model.fit(X_train, y_train); train_time=time.time()-start
    y_pred=model.predict(X_test); y_prob=model.predict_proba(X_test)[:,1]
    results_all.append({'Model':name,'Feature':'Structured','Accuracy':accuracy_score(y_test,y_pred),
        'Precision':precision_score(y_test,y_pred),'Recall':recall_score(y_test,y_pred),
        'F1':f1_score(y_test,y_pred),'AUC':roc_auc_score(y_test,y_prob),'Time(s)':round(train_time,2)})
    print(f'  {name}: F1={results_all[-1]["F1"]*100:.2f}%')

# --- Combined models ---
comb_models = {
    'LR (Combined)': LogisticRegression(max_iter=1000, random_state=42, class_weight='balanced'),
    'SVM (Combined)': CalibratedClassifierCV(LinearSVC(max_iter=2000, random_state=42, class_weight='balanced'), cv=3),
}
for name, model in comb_models.items():
    start=time.time(); model.fit(X_combined_train, y_train); train_time=time.time()-start
    y_pred=model.predict(X_combined_test); y_prob=model.predict_proba(X_combined_test)[:,1]
    results_all.append({'Model':name,'Feature':'Combined','Accuracy':accuracy_score(y_test,y_pred),
        'Precision':precision_score(y_test,y_pred),'Recall':recall_score(y_test,y_pred),
        'F1':f1_score(y_test,y_pred),'AUC':roc_auc_score(y_test,y_prob),'Time(s)':round(train_time,2)})
    print(f'  {name}: F1={results_all[-1]["F1"]*100:.2f}%')

comp_df = pd.DataFrame(results_all).sort_values('F1', ascending=False).reset_index(drop=True)
print(f'\nTotal models compared: {len(comp_df)}')

## Section 3: Results Table

In [ ]:
# Full comparison table
print('=== COMPARATIVE ANALYSIS RESULTS ===\n')
disp=comp_df.copy()
for c in ['Accuracy','Precision','Recall','F1']: disp[c]=(disp[c]*100).round(2).astype(str)+'%'
disp['AUC']=disp['AUC'].round(4)
print(disp.to_string(index=False))
print(f'\nBest model: {comp_df.iloc[0]["Model"]} (F1={comp_df.iloc[0]["F1"]*100:.2f}%)')

## Section 4: Visualization

In [ ]:
# Multi-metric comparison chart
fig,axes=plt.subplots(2,2,figsize=(16,12))

# F1 by model
sns.barplot(data=comp_df,y='Model',x='F1',hue='Feature',ax=axes[0,0],palette='Set2')
axes[0,0].set_title('F1 Score by Model');axes[0,0].set_xlim(0,1)

# AUC by model
sns.barplot(data=comp_df,y='Model',x='AUC',hue='Feature',ax=axes[0,1],palette='Set2')
axes[0,1].set_title('AUC-ROC by Model');axes[0,1].set_xlim(0,1)

# Training time
sns.barplot(data=comp_df,y='Model',x='Time(s)',hue='Feature',ax=axes[1,0],palette='Set2')
axes[1,0].set_title('Training Time (seconds)')

# Radar-style: top 5 models
top5=comp_df.head(5)
metrics=['Accuracy','Precision','Recall','F1','AUC']
x=np.arange(len(metrics))
w=0.15
for i,(_,row) in enumerate(top5.iterrows()):
    vals=[row[m] for m in metrics]
    axes[1,1].bar(x+i*w,vals,w,label=row['Model'])
axes[1,1].set_xticks(x+w*2);axes[1,1].set_xticklabels(metrics)
axes[1,1].set_title('Top 5 Models - All Metrics');axes[1,1].legend(fontsize=7);axes[1,1].set_ylim(0,1)

plt.tight_layout();plt.savefig('../models/comparative_analysis.png',dpi=150,bbox_inches='tight');plt.show()

## Section 5: Statistical Validation

In [ ]:
# McNemar's test for pairwise model comparison
from scipy.stats import chi2

print('=== McNemars Test (Top 3 models) ===\n')
# Retrain top 3 to get predictions
top3_names = comp_df.head(3)['Model'].tolist()
print(f'Comparing: {top3_names}')

# Use stored predictions from above for comparison
# Friedman test across CV folds
print('\n=== Friedman Test (CV Folds) ===')
from scipy.stats import friedmanchisquare
cv5=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
fold_scores={}
test_models={'LR':LogisticRegression(max_iter=1000,random_state=42,class_weight='balanced'),
    'XGB':XGBClassifier(n_estimators=300,max_depth=6,learning_rate=0.1,scale_pos_weight=scale_weight,random_state=42,eval_metric='logloss',use_label_encoder=False),
    'GB':GradientBoostingClassifier(n_estimators=200,max_depth=5,random_state=42)}
for name,model in test_models.items():
    X_cv=X_combined_train if name=='LR' else X_train
    scores=cross_val_score(model,X_cv,y_train,cv=cv5,scoring='f1',n_jobs=-1)
    fold_scores[name]=scores
    print(f'  {name}: {scores.round(4)}')

stat,p=friedmanchisquare(*fold_scores.values())
print(f'\nFriedman chi2={stat:.4f}, p={p:.4f}')
print('Significant difference' if p<0.05 else 'No significant difference')

comp_df.to_csv('../models/comparative_results.csv',index=False)
print('\nResults saved.')